In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
import os
import seaborn as sns

sns.set_context("notebook")
# --- Clinical palette (blue + teal) ---
CLINICAL_COLORS = {
    "ViennaAIdb": "#005B96",  # Deep Clinical Blue
    "MIMIC": "#00A6A6",       # Teal
    "Reference": "black"
}


def calculate_auc_ci(y_true, y_pred, n_bootstraps=20, rng_seed=42):
    """
    Calculates the 95% Confidence Interval for AUC using bootstrapping.
    Returns: (auc_score, lower_bound, upper_bound, tpr_lower, tpr_upper, mean_fpr)
    """
    rng = np.random.RandomState(rng_seed)
    bootstrapped_auc = []
    tprs = []
    base_fpr = np.linspace(0, 1, 101)

    # Calculate original AUC
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    original_auc = auc(fpr, tpr)

    print(f"Bootstrapping CI for AUC ({n_bootstraps} iterations)...")
    
    for i in range(n_bootstraps):
        # Bootstrap by sampling with replacement
        indices = rng.randint(0, len(y_pred), len(y_pred))
        
        if len(np.unique(y_true[indices])) < 2:
            continue
        print(len(np.unique(indices)), len(y_true))
        fpr_boot, tpr_boot, _ = roc_curve(y_true[np.unique(indices)], y_pred[np.unique(indices)])
        score = auc(fpr_boot, tpr_boot)
        bootstrapped_auc.append(score)
        
        # Interpolate TPR for confidence band
        tpr_interp = np.interp(base_fpr, fpr_boot, tpr_boot)
        tpr_interp[0] = 0.0
        tprs.append(tpr_interp)

    sorted_scores = np.array(bootstrapped_auc)
    sorted_scores.sort()

    # 95% CI for AUC score
    lower_bound = sorted_scores[int(0.025 * len(sorted_scores))]
    upper_bound = sorted_scores[int(0.975 * len(sorted_scores))]
    
    # 95% CI for TPR (Confidence Band)
    tprs = np.array(tprs)
    tpr_lower = np.percentile(tprs, 2.5, axis=0)
    tpr_upper = np.percentile(tprs, 97.5, axis=0)

    return original_auc, lower_bound, upper_bound, tpr_lower, tpr_upper, base_fpr

def plot_abstract_figure():
    # --- Configuration ---
    output_dir = 'model_outputs'
    muw_file = os.path.join(output_dir, 'muw_results.npz')
    mimic_file = os.path.join(output_dir, 'mimic_results.npz')
    save_path = 'ROC_with_CI.png'

    # --- Load Data ---
    data_loaded = {}
    
    for name, filepath in [('Internal', muw_file), ('External', mimic_file)]:
        if os.path.exists(filepath):
            data = np.load(filepath)
            data_loaded[name] = {
                'y_true': data['y_true'],
                'y_pred': data['y_pred_proba'],
            }
            print(f"Loaded {name} data from {filepath}")
        else:
            print(f"Warning: {filepath} not found.")

    if not data_loaded:
        print("No data found. Please run the save scripts in your notebooks.")
        return

    sns.set_theme(style="whitegrid", palette="pastel")
    fig, ax = plt.subplots()
    
    # Colors from Seaborn's colorblind palette
    colors = sns.color_palette("colorblind")

    # --- 1. Internal Validation (MUW) ---
    if 'Internal' in data_loaded:
        d = data_loaded['Internal']
        # Calculate stats
        auc_score, low, high, tpr_low, tpr_high, mean_fpr = calculate_auc_ci(d['y_true'], d['y_pred'])
        
        # Get Original Curve
        fpr, tpr, _ = roc_curve(d['y_true'], d['y_pred'])
        
        # Plot Curve
        ax.plot(fpr, tpr,
                color=CLINICAL_COLORS['ViennaAIdb'],
                lw=1,
                label=f'ViennaAIdb (AUC: {auc_score:.2f} [{low:.2f}-{high:.2f}])')
        ax.fill_between(mean_fpr, tpr_low, tpr_high, color=CLINICAL_COLORS['ViennaAIdb'], alpha=0.2)
        


    # --- 2. External Validation (MIMIC) ---
    if 'External' in data_loaded:
        d = data_loaded['External']
        # Calculate stats
        auc_score, low, high, tpr_low, tpr_high, mean_fpr = calculate_auc_ci(d['y_true'], d['y_pred'])
        
        # Get Original Curve
        fpr, tpr, _ = roc_curve(d['y_true'], d['y_pred'])
        
        # Plot Curve
        ax.plot(fpr, tpr,
                color=CLINICAL_COLORS['MIMIC'],
                lw=1,
                label=f'MIMIC (AUC: {auc_score:.2f} [{low:.2f}-{high:.2f}])')
        ax.fill_between(mean_fpr, tpr_low, tpr_high, color=CLINICAL_COLORS['MIMIC'], alpha=0.2)
        

    # --- Formatting ---
    ax.plot([0, 1], [0, 1], color='black', lw=1, linestyle=':', alpha=0.5)
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.0])
    ax.set_aspect('equal', 'box')
    
    # Seaborn styling handles grid, but let's ensure labels are bold/large
    ax.set_xlabel('1 - Specificity (False Positive Rate)')
    ax.set_ylabel('Sensitivity (True Positive Rate)')

    # Legend
    ax.legend(loc="lower right", fontsize=10, frameon=True, edgecolor='black', fancybox=False)
    
    # --- Save ---
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Figure with CI bands saved to {save_path}")
    plt.show()

if __name__ == "__main__":
    plot_abstract_figure()